In [1]:
# %% 셀 1: eval.jsonl 파싱 + 채널 통계 (나중에 bucket별 분석용)
import os
import re
import json
import glob
import random
import hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
INST_DIR   = "./data/6_telop_instances"

N_BUCKETS  = 5  # 나중에 평가 시 bucket별 분석용
SEED       = 42


# -------------------------
# 1) eval.jsonl에서 전체 채널 목록 추출
# -------------------------
def extract_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


eval_channels: set[str] = set()
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    user_msg = next(m for m in ex["messages"] if m["role"] == "user")
    text = extract_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", text)
    if m_ch:
        eval_channels.add(m_ch.group(1).strip())

all_channels = sorted(eval_channels)
print(f"\neval.jsonl의 고유 채널 수: {len(all_channels)}")


# -------------------------
# 2) 채널별 통계 (나중에 bucket별 분석용)
# -------------------------
def channel_stats(channel: str) -> dict | None:
    ch_dir = os.path.join(INST_DIR, channel)
    if not os.path.isdir(ch_dir):
        return None

    json_paths = sorted(glob.glob(os.path.join(glob.escape(ch_dir), "*.json")))
    if not json_paths:
        return None

    n_videos    = 0
    n_instances = 0
    dur_sum     = 0.0
    textlen_sum = 0
    density_sum = 0.0

    for jp in json_paths:
        try:
            with open(jp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        instances = data.get("instances", [])
        n_videos += 1

        if not instances:
            continue

        starts = np.array([inst["start_sec"] for inst in instances], dtype=np.float64)
        ends   = np.array([inst["end_sec"]   for inst in instances], dtype=np.float64)
        texts  = [inst["text"] for inst in instances]

        durs    = np.clip(ends - starts, 0, None)
        vid_len = float(ends.max()) if len(ends) else 0.0

        n_instances += len(instances)
        dur_sum     += float(durs.sum())
        textlen_sum += sum(len(t) for t in texts)

        if vid_len > 0:
            density_sum += float(durs.sum()) / vid_len

    if n_videos == 0:
        return None

    return {
        "n_videos":     n_videos,
        "n_instances":  n_instances,
        "avg_count":    n_instances / n_videos,
        "avg_duration": (dur_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_text_len": (textlen_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_density":  density_sum / n_videos,
    }


stats_rows = []
for ch in tqdm(all_channels, desc="채널 통계 계산"):
    s = channel_stats(ch)
    if s is None:
        continue
    stats_rows.append({"channel": ch, **s})

stats_df = pd.DataFrame(stats_rows)
stats_df = stats_df.sort_values("avg_density").reset_index(drop=True)
stats_df["bucket"] = pd.qcut(
    stats_df["avg_density"],
    q=N_BUCKETS,
    labels=list(range(N_BUCKETS)),
).astype(int)

print(f"\n✅ 전체 채널: {len(stats_df)}개")
print(stats_df.groupby("bucket")["avg_density"].agg(["count", "min", "max", "mean"]))


# -------------------------
# 3) 영상별 swap 채널 미리 결정 (재현성 보장)
# -------------------------
def get_swap_channel(orig_channel: str, video_name: str, candidates: list[str]) -> str:
    """hash 기반으로 결정론적 swap 채널 선택. 자기 자신 제외."""
    h = hashlib.sha256(f"{orig_channel}||{video_name}".encode()).hexdigest()
    seed = int(h[:8], 16)
    rng = random.Random(seed)
    pool = [c for c in candidates if c != orig_channel]
    return rng.choice(pool)


# 테스트
_test_ch = all_channels[0]
_test_vn = "test_video"
_test_swap = get_swap_channel(_test_ch, _test_vn, all_channels)
print(f"\nswap 테스트: {_test_ch!r} → {_test_swap!r}")

print(f"\n✅ 다음 셀에서 사용할 변수: all_channels, stats_df, get_swap_channel")

/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 63840.54it/s]



eval.jsonl의 고유 채널 수: 664


채널 통계 계산: 100%|██████████| 664/664 [00:21<00:00, 30.63it/s]


✅ 전체 채널: 664개
        count       min        max      mean
bucket                                      
0         133  0.011472   0.664083  0.294273
1         133  0.680476   1.685595  1.158782
2         132  1.686024   3.237476  2.516964
3         133  3.251788   4.644189  3.934458
4         133  4.646865  16.830567  6.211632

swap 테스트: '\x08고기남자' → 'MBN MUSIC'

✅ 다음 셀에서 사용할 변수: all_channels, stats_df, get_swap_channel


In [2]:
# %% 셀 2: 기존 sglang 서버 종료 + ep3 merged 모델 서버 시작
import os
import subprocess
import time
import requests
import signal

os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

MODEL_PATH      = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"
PORT            = 8000
SERVER_LOG      = "sglang_server.log"
CONTEXT_LENGTH  = 131072


# -------------------------
# 1) 기존 서버 프로세스 종료
# -------------------------
print("🛑 기존 sglang 서버 종료 시도...")

if "server_process" in globals() and server_process.poll() is None:
    try:
        server_process.terminate()
        server_process.wait(timeout=10)
        print("   ✅ 이전 server_process 핸들로 종료")
    except Exception as e:
        try:
            server_process.kill()
            print(f"   ⚠️ terminate 실패 → kill ({e})")
        except Exception:
            pass

try:
    out = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    pids = [int(p) for p in out.splitlines() if p.strip().isdigit()]
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"   ✅ PID {pid} SIGTERM")
        except ProcessLookupError:
            pass
    time.sleep(3)
    out2 = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    for p in out2.splitlines():
        if p.strip().isdigit():
            try:
                os.kill(int(p), signal.SIGKILL)
                print(f"   ✅ PID {p} SIGKILL")
            except ProcessLookupError:
                pass
except FileNotFoundError:
    print("   ⚠️ lsof 없음 — 포트 점유 확인 skip")

for _ in range(10):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            time.sleep(1)
            continue
    except Exception:
        break
print("   ✅ 포트 정리 완료\n")


# -------------------------
# 2) 서버 시작
# -------------------------
assert os.path.isdir(MODEL_PATH), f"모델 경로 없음: {MODEL_PATH}"

sglang_log = open(SERVER_LOG, "w")

server_process = subprocess.Popen(
    [
        "python", "-m", "sglang.launch_server",
        "--model-path", MODEL_PATH,
        "--port", str(PORT),
        "--mem-fraction-static", "0.8",
        "--context-length", str(CONTEXT_LENGTH),
        "--reasoning-parser", "qwen3",
        "--attention-backend", "triton",
    ],
    stdout=sglang_log,
    stderr=subprocess.STDOUT,
)


print(f"⏳ SGLang 서버 시작 중...")
print(f"   모델: {MODEL_PATH}")
print(f"   context-length: {CONTEXT_LENGTH}")

ready = False
for i in range(600):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            print(f"✅ 서버 준비 완료! ({i}초)")
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print(f"❌ 서버 시작 실패. 로그 확인: tail -n 200 {SERVER_LOG}")

🛑 기존 sglang 서버 종료 시도...
   ✅ 포트 정리 완료

⏳ SGLang 서버 시작 중...
   모델: ./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged
   context-length: 131072
✅ 서버 준비 완료! (24초)


In [4]:
# %% 셀 3: 664채널 × 5영상 × (본채널 + 랜덤1채널) inference
import os
import re
import json
import time
import httpx
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

EVAL_JSONL  = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR  = "./data/7_qwen_test"
MODEL_NAME  = "./model/qwen-finetune-models/qwen3.5-9b-lora-teloptextgen-rank64-split-16K-datacleaning-ep3-merged"

TEMPERATURE = 0.1
TOP_P       = 1.0
MAX_TOKENS  = 126976
PRESENCE_PENALTY = 1.5
FREQUENCY_PENALTY = 0.1

MAX_WORKERS = 20
TIMEOUT     = 3600

os.makedirs(OUTPUT_DIR, exist_ok=True)

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(TIMEOUT, connect=5.0),
)


# -------------------------
# 1) eval.jsonl 전체 파싱
# -------------------------
def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    eval_lines = f.readlines()

target_entries = []
for line in tqdm(eval_lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    msgs = ex["messages"]
    sys_msg  = next(m for m in msgs if m["role"] == "system")
    user_msg = next(m for m in msgs if m["role"] == "user")
    user_text = parse_user_text(user_msg["content"])

    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    channel = m_ch.group(1).strip()
    video   = m_vn.group(1).strip()

    target_entries.append({
        "channel":    channel,
        "video_name": video,
        "system":     sys_msg["content"],
        "user_text":  user_text,
    })

print(f"\n✅ 대상 영상 수: {len(target_entries)}")


# -------------------------
# 2) job 리스트 구성: 영상당 2건 (본채널 + 랜덤1채널)
# -------------------------
def swap_channel_line(user_text: str, new_channel: str) -> str:
    return re.sub(
        r"채널:\s*[^\n]+",
        f"채널: {new_channel}",
        user_text,
        count=1,
    )


def output_path(orig_ch: str, video: str, cond_ch: str) -> str:
    return os.path.join(OUTPUT_DIR, orig_ch, video, f"{cond_ch}.json")


jobs = []
for ent in tqdm(target_entries, desc="job 리스트 구성"):
    orig_ch = ent["channel"]
    video   = ent["video_name"]

    # (a) 본채널 조건
    out_a = output_path(orig_ch, video, orig_ch)
    if not os.path.exists(out_a):
        jobs.append({
            "orig_channel": orig_ch,
            "video_name":   video,
            "cond_channel": orig_ch,
            "system":       ent["system"],
            "user_text":    ent["user_text"],  # 원본 그대로
            "out_path":     out_a,
        })
    
    # (b) 랜덤 swap 채널 조건
    swap_ch = get_swap_channel(orig_ch, video, all_channels)
    out_b = output_path(orig_ch, video, swap_ch)
    if not os.path.exists(out_b):
        jobs.append({
            "orig_channel": orig_ch,
            "video_name":   video,
            "cond_channel": swap_ch,
            "system":       ent["system"],
            "user_text":    swap_channel_line(ent["user_text"], swap_ch),
            "out_path":     out_b,
        })
    

total_expected = len(target_entries) * 2
print(f"\n📋 실행할 job: {len(jobs)} (skip: {total_expected - len(jobs)})")

# -------------------------
# 3) 단일 요청 처리
# -------------------------
def call_model(job: dict) -> dict:
    t0 = time.time()
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": job["system"]},
                {"role": "user",   "content": job["user_text"]},
            ],
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            presence_penalty=PRESENCE_PENALTY,
            frequency_penalty=FREQUENCY_PENALTY,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0

        raw = resp.choices[0].message.content.strip()
        cleaned = raw
        if cleaned.startswith("```"):
            cleaned = re.sub(r"^```[a-zA-Z]*\n?", "", cleaned)
            cleaned = re.sub(r"\n?```$", "", cleaned)

        parsed = json.loads(cleaned)

        os.makedirs(os.path.dirname(job["out_path"]), exist_ok=True)
        record = {
            "orig_channel":      job["orig_channel"],
            "video_name":        job["video_name"],
            "cond_channel":      job["cond_channel"],
            "model":             MODEL_NAME,
            "params": {
                "temperature":   TEMPERATURE,
                "top_p":         TOP_P,
                "max_tokens":    MAX_TOKENS,
            },
            "system":            job["system"],
            "user_text":         job["user_text"],
            "raw_output":        raw,
            "instances":         parsed.get("instances", []) if isinstance(parsed, dict) else parsed,
            "elapsed_sec":       round(elapsed, 2),
            "prompt_tokens":     getattr(resp.usage, "prompt_tokens", None),
            "completion_tokens": getattr(resp.usage, "completion_tokens", None),
        }
        with open(job["out_path"], "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        return {"success": True, "elapsed": elapsed}

    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSONDecodeError: {str(e)[:100]}",
                "elapsed": time.time() - t0}
    except Exception as e:
        return {"success": False, "error": str(e)[:200],
                "elapsed": time.time() - t0}


# -------------------------
# 4) 병렬 실행
# -------------------------
def classify_error(err_msg: str) -> str:
    """에러 메시지를 카테고리로 분류"""
    if err_msg.startswith("JSONDecodeError"):
        return "json"
    low = err_msg.lower()
    if "readtimeout" in low or "read timeout" in low or "timed out" in low:
        return "timeout"
    if "connecterror" in low or "connection" in low or "remoteprotocol" in low:
        return "conn"
    if "contextlength" in low or "context length" in low or "maximum context" in low or "token" in low and "exceed" in low:
        return "ctxlen"
    if "500" in err_msg or "internal server" in low or "cuda" in low or "out of memory" in low or "oom" in low:
        return "server"
    if "400" in err_msg or "badrequest" in low or "bad request" in low:
        return "badreq"
    if "permission" in low or "no such file" in low or "disk" in low or "oserror" in low or "filenotfound" in low:
        return "io"
    if "indexerror" in low or "attributeerror" in low or "keyerror" in low or "typeerror" in low:
        return "resp"
    return "other"


if not jobs:
    print("\n✅ 모든 job이 이미 완료 — 새로 생성할 것 없음")
else:
    n_success = 0
    fail_counts = {
        "json":    0,  # JSONDecodeError (파싱 실패, max_tokens 도달 등)
        "timeout": 0,  # httpx ReadTimeout
        "conn":    0,  # 연결 끊김 / ConnectError
        "ctxlen":  0,  # context length 초과
        "server":  0,  # 서버 500 / CUDA OOM
        "badreq":  0,  # 400 Bad Request
        "io":      0,  # 파일 I/O 에러
        "resp":    0,  # resp 구조 이상 (IndexError 등)
        "other":   0,  # 기타
    }
    errors    = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(call_model, j): j for j in jobs}

        with tqdm(total=len(jobs), desc="inference") as pbar:
            for fut in as_completed(futures):
                job = futures[fut]
                res = fut.result()
                if res["success"]:
                    n_success += 1
                else:
                    cat = classify_error(res["error"])
                    fail_counts[cat] += 1
                    errors.append({
                        "orig":  job["orig_channel"],
                        "video": job["video_name"],
                        "cond":  job["cond_channel"],
                        "cat":   cat,
                        "err":   res["error"],
                    })
                pbar.update(1)
                pbar.set_postfix(
                    ok=n_success,
                    **{k: v for k, v in fail_counts.items() if v > 0},
                )

    n_fail = sum(fail_counts.values())
    print(f"\n📊 완료")
    print(f"  성공: {n_success}")
    print(f"  실패: {n_fail}")
    if n_fail > 0:
        print(f"\n  실패 원인별:")
        labels = {
            "json":    "JSON 파싱 실패",
            "timeout": "Read timeout",
            "conn":    "연결 에러",
            "ctxlen":  "Context length 초과",
            "server":  "서버 500 / OOM",
            "badreq":  "400 Bad request",
            "io":      "파일 I/O 에러",
            "resp":    "응답 구조 이상",
            "other":   "기타",
        }
        for k, v in fail_counts.items():
            if v > 0:
                print(f"    {labels[k]:20s}: {v}")
    if errors:
        print(f"\n실패 샘플 (최대 10개):")
        for e in errors[:10]:
            print(f"  [{e['cat']}][{e['orig']}] {e['video']} ← {e['cond']}: {e['err']}")

eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 59735.87it/s]



✅ 대상 영상 수: 3320


job 리스트 구성: 100%|██████████| 3320/3320 [00:00<00:00, 44159.39it/s]



📋 실행할 job: 1 (skip: 6639)


inference: 100%|██████████| 1/1 [00:11<00:00, 11.17s/it, ok=1]


📊 완료
  성공: 1
  실패: 0


In [13]:
# %% 텔롭 텍스트 생성 품질 평가 — 존재/개수/텍스트 3분리
import os, json, glob, re
import numpy as np
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from scipy.optimize import linear_sum_assignment
from concurrent.futures import ProcessPoolExecutor, as_completed

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"
BIN_SIZE = 0.1


def normalize_text(text):
    return re.sub(r'\s+', '', str(text))

def text_sim(t1, t2):
    n1, n2 = normalize_text(t1), normalize_text(t2)
    if not n1 and not n2:
        return 1.0
    if not n1 or not n2:
        return 0.0
    if n1 == n2:
        return 1.0
    return SequenceMatcher(None, n1, n2).ratio()

def instances_to_text_bins(instances, bin_size=BIN_SIZE):
    bins = {}
    for inst in instances:
        s, e = inst["start_sec"], inst["end_sec"]
        b_start = int(s / bin_size)
        b_end = max(b_start + 1, int(e / bin_size))
        text = inst.get("text", "")
        for b in range(b_start, b_end):
            if b not in bins:
                bins[b] = []
            bins[b].append(text)
    return bins

def bin_scores(gt_texts, pred_texts):
    n_g, n_p = len(gt_texts), len(pred_texts)
    if n_g == 0 and n_p == 0:
        return 1.0, 1.0, 1.0, "silent"
    if n_g == 0:
        return 0.0, 0.0, 0.0, "halluc"
    if n_p == 0:
        return 0.0, 0.0, 0.0, "miss"
    count = min(n_g, n_p) / max(n_g, n_p)
    cost = np.zeros((n_g, n_p), dtype=np.float64)
    for i, g in enumerate(gt_texts):
        for j, p in enumerate(pred_texts):
            cost[i, j] = text_sim(g, p)
    row_ind, col_ind = linear_sum_assignment(-cost)
    text_q = cost[row_ind, col_ind].sum() / min(n_g, n_p)
    return 1.0, count, text_q, "both"

def compute_video_score(gt_instances, pred_instances):
    gt_bins = instances_to_text_bins(gt_instances)
    pred_bins = instances_to_text_bins(pred_instances)
    all_bin_keys = set(gt_bins.keys()) | set(pred_bins.keys())
    if not all_bin_keys:
        return {k: 1.0 for k in [
            "exist_all","exist_active","count_all","count_active","count_both",
            "text_all","text_active","text_both",
            "n_all","n_active","n_both","n_silent","n_miss","n_halluc"]}
    max_bin = max(all_bin_keys)
    total_bins = max_bin + 1
    exist_all, count_all, text_all = [], [], []
    exist_active, count_active, text_active = [], [], []
    count_both, text_both = [], []
    n_silent = n_miss = n_halluc = n_both = 0
    for b in range(total_bins):
        gt_t = gt_bins.get(b, [])
        pred_t = pred_bins.get(b, [])
        e, c, t, kind = bin_scores(gt_t, pred_t)
        exist_all.append(e); count_all.append(c); text_all.append(t)
        if kind == "silent":
            n_silent += 1
        else:
            exist_active.append(e); count_active.append(c); text_active.append(t)
            if kind == "miss":
                n_miss += 1
            elif kind == "halluc":
                n_halluc += 1
            else:
                n_both += 1
                count_both.append(c); text_both.append(t)
    def m(arr, default=1.0):
        return float(np.mean(arr)) if arr else default
    return {
        "exist_all": m(exist_all), "exist_active": m(exist_active),
        "count_all": m(count_all), "count_active": m(count_active), "count_both": m(count_both),
        "text_all": m(text_all), "text_active": m(text_active), "text_both": m(text_both),
        "n_all": len(exist_all), "n_active": len(exist_active), "n_both": n_both,
        "n_silent": n_silent, "n_miss": n_miss, "n_halluc": n_halluc,
    }

def compute_one(args):
    channel, video, n_gt, n_pred, gt_instances, pred_instances = args
    m = compute_video_score(gt_instances, pred_instances)
    return {"channel": channel, "video": video, "n_gt": n_gt, "n_pred": n_pred, **m}


# ═══════════════════════════════════════
# 1) Pred 수집 (본채널만)
# ═══════════════════════════════════════
pred_map = {}
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="Pred 수집"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    if rec.get("orig_channel") != rec.get("cond_channel"):
        continue
    key = (rec["orig_channel"], rec["video_name"])
    pred_map[key] = rec.get("instances", [])

# ═══════════════════════════════════════
# 2) GT 수집 + task 구성 (JSONL 줄 단위 순회)
# ═══════════════════════════════════════
tasks = []
with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        msgs = ex["messages"]
        user_msg = next(m for m in msgs if m["role"] == "user")
        asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst_msg is None:
            continue
        user_text = user_msg["content"] if isinstance(user_msg["content"], str) else ""
        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn:
            continue
        key = (m_ch.group(1).strip(), m_vn.group(1).strip())
        if key not in pred_map:
            continue
        try:
            gt_instances = json.loads(asst_msg["content"]).get("instances", [])
        except:
            gt_instances = []
        tasks.append((key[0], key[1], len(gt_instances), len(pred_map[key]),
                       gt_instances, pred_map[key]))

print(f"🎯 {len(tasks):,}개 영상 평가")

# ═══════════════════════════════════════
# 3) 병렬 실행
# ═══════════════════════════════════════
results = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(compute_one, t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures), desc="메트릭 계산"):
        results.append(f.result())

print(f"\n✅ 완료: {len(results):,}개")

# ═══════════════════════════════════════
# 결과 출력
# ═══════════════════════════════════════
gt_ns = np.array([r["n_gt"] for r in results])
pred_ns = np.array([r["n_pred"] for r in results])

metrics = {}
for key in ["exist_all","exist_active","count_all","count_active","count_both",
            "text_all","text_active","text_both"]:
    metrics[key] = np.array([r[key] for r in results])

print(f"\n📊 텔롭 텍스트 생성 품질 — 존재/개수/텍스트 × 전체/활성/양쪽")
print(f"  {'메트릭':<20} {'전체(all)':>10} {'활성(act)':>10} {'양쪽(both)':>10}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*10}")
print(f"  {'존재':<20} {metrics['exist_all'].mean():>10.3f} {metrics['exist_active'].mean():>10.3f} {'—':>10}")
print(f"  {'개수':<20} {metrics['count_all'].mean():>10.3f} {metrics['count_active'].mean():>10.3f} {metrics['count_both'].mean():>10.3f}")
print(f"  {'텍스트':<20} {metrics['text_all'].mean():>10.3f} {metrics['text_active'].mean():>10.3f} {metrics['text_both'].mean():>10.3f}")

print(f"\n  (median)")
print(f"  {'메트릭':<20} {'전체(all)':>10} {'활성(act)':>10} {'양쪽(both)':>10}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*10}")
print(f"  {'존재':<20} {np.median(metrics['exist_all']):>10.3f} {np.median(metrics['exist_active']):>10.3f} {'—':>10}")
print(f"  {'개수':<20} {np.median(metrics['count_all']):>10.3f} {np.median(metrics['count_active']):>10.3f} {np.median(metrics['count_both']):>10.3f}")
print(f"  {'텍스트':<20} {np.median(metrics['text_all']):>10.3f} {np.median(metrics['text_active']):>10.3f} {np.median(metrics['text_both']):>10.3f}")

# ── bin 유형 통계 ──
total_all = sum(r["n_all"] for r in results)
total_silent = sum(r["n_silent"] for r in results)
total_active = sum(r["n_active"] for r in results)
total_miss = sum(r["n_miss"] for r in results)
total_halluc = sum(r["n_halluc"] for r in results)
total_both = sum(r["n_both"] for r in results)

print(f"\n📊 bin 유형 통계")
print(f"  전체 bin: {total_all:,}")
print(f"  정적 (양쪽 다 없음): {total_silent:,} ({total_silent/total_all*100:.1f}%)")
print(f"  활성 (한쪽 이상 있음): {total_active:,} ({total_active/total_all*100:.1f}%)")
print(f"    양쪽 다 있음: {total_both:,} ({total_both/max(total_active,1)*100:.1f}%)")
print(f"    누락 (GT有, pred無): {total_miss:,} ({total_miss/max(total_active,1)*100:.1f}%)")
print(f"    환각 (GT無, pred有): {total_halluc:,} ({total_halluc/max(total_active,1)*100:.1f}%)")

# ── GT 인스턴스 구간별 ──
print(f"\n📊 GT 인스턴스 구간별 (mean)")
print(f"  {'GT 구간':<15} {'n':>6} {'존전체':>7} {'존활성':>7} {'개활성':>7} {'개양쪽':>7} {'텍활성':>7} {'텍양쪽':>7}")
print(f"  {'─'*15} {'─'*6} {'─'*7} {'─'*7} {'─'*7} {'─'*7} {'─'*7} {'─'*7}")
for lo, hi in [(0,1),(1,10),(10,100),(100,1000),(1000,10000)]:
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0:
        continue
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6}"
          f" {metrics['exist_all'][mask].mean():>7.3f}"
          f" {metrics['exist_active'][mask].mean():>7.3f}"
          f" {metrics['count_active'][mask].mean():>7.3f}"
          f" {metrics['count_both'][mask].mean():>7.3f}"
          f" {metrics['text_active'][mask].mean():>7.3f}"
          f" {metrics['text_both'][mask].mean():>7.3f}")

# ── GT=0 vs GT>0 ──
gt_zero = gt_ns == 0
gt_nonzero = gt_ns > 0
print(f"\n📊 GT=0: {gt_zero.sum()}개")
n_both_zero = ((gt_zero) & (pred_ns == 0)).sum()
n_halluc_vid = ((gt_zero) & (pred_ns > 0)).sum()
print(f"  양쪽 다 없음: {n_both_zero} ({n_both_zero/max(gt_zero.sum(),1)*100:.1f}%)")
print(f"  환각 (pred>0): {n_halluc_vid} ({n_halluc_vid/max(gt_zero.sum(),1)*100:.1f}%)")

if gt_nonzero.sum() > 0:
    print(f"\n📊 GT>0: {gt_nonzero.sum()}개")
    for key, label in [("exist_all","존재 전체"),("exist_active","존재 활성"),
                        ("count_active","개수 활성"),("count_both","개수 양쪽"),
                        ("text_active","텍스트 활성"),("text_both","텍스트 양쪽")]:
        arr = metrics[key][gt_nonzero]
        print(f"  {label:<12}: mean={arr.mean():.3f}, median={np.median(arr):.3f}")

Pred 수집: 100%|██████████| 6630/6630 [00:01<00:00, 6221.07it/s]


🎯 3,320개 영상 평가


메트릭 계산: 100%|██████████| 3320/3320 [00:03<00:00, 915.74it/s] 


✅ 완료: 3,320개

📊 텔롭 텍스트 생성 품질 — 존재/개수/텍스트 × 전체/활성/양쪽
  메트릭                     전체(all)    활성(act)   양쪽(both)
  ──────────────────── ────────── ────────── ──────────
  존재                        0.829      0.712          —
  개수                        0.631      0.511      0.754
  텍스트                       0.521      0.399      0.567

  (median)
  메트릭                     전체(all)    활성(act)   양쪽(both)
  ──────────────────── ────────── ────────── ──────────
  존재                        0.983      0.975          —
  개수                        0.673      0.574      0.783
  텍스트                       0.529      0.382      0.568

📊 bin 유형 통계
  전체 bin: 1,327,759.0
  정적 (양쪽 다 없음): 103,079.0 (7.8%)
  활성 (한쪽 이상 있음): 1,224,836.0 (92.2%)
    양쪽 다 있음: 1,030,971.0 (84.2%)
    누락 (GT有, pred無): 61,439.0 (5.0%)
    환각 (GT無, pred有): 132,738.0 (10.8%)

📊 GT 인스턴스 구간별 (mean)
  GT 구간                n     존전체     존활성     개활성     개양쪽     텍활성     텍양쪽
  ─────────────── ────── ─────── ─────── ─────── ─────── ─────── ─

In [14]:
# %% 상대적 비교: pred_orig vs pred_swap — 존재/개수/텍스트 3분리
import os, json, glob, re
import numpy as np
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from scipy.optimize import linear_sum_assignment
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.stats import wilcoxon

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_test"
BIN_SIZE = 0.1

def normalize_text(text):
    return re.sub(r'\s+', '', str(text))

def text_sim(t1, t2):
    n1, n2 = normalize_text(t1), normalize_text(t2)
    if not n1 and not n2:
        return 1.0
    if not n1 or not n2:
        return 0.0
    if n1 == n2:
        return 1.0
    return SequenceMatcher(None, n1, n2).ratio()

def instances_to_text_bins(instances, bin_size=BIN_SIZE):
    bins = {}
    for inst in instances:
        s, e = inst["start_sec"], inst["end_sec"]
        b_start = int(s / bin_size)
        b_end = max(b_start + 1, int(e / bin_size))
        text = inst.get("text", "")
        for b in range(b_start, b_end):
            if b not in bins:
                bins[b] = []
            bins[b].append(text)
    return bins

def bin_scores(gt_texts, pred_texts):
    n_g, n_p = len(gt_texts), len(pred_texts)
    if n_g == 0 and n_p == 0:
        return 1.0, 1.0, 1.0, "silent"
    if n_g == 0:
        return 0.0, 0.0, 0.0, "halluc"
    if n_p == 0:
        return 0.0, 0.0, 0.0, "miss"
    count = min(n_g, n_p) / max(n_g, n_p)
    cost = np.zeros((n_g, n_p), dtype=np.float64)
    for i, g in enumerate(gt_texts):
        for j, p in enumerate(pred_texts):
            cost[i, j] = text_sim(g, p)
    row_ind, col_ind = linear_sum_assignment(-cost)
    text_q = cost[row_ind, col_ind].sum() / min(n_g, n_p)
    return 1.0, count, text_q, "both"

def compute_video_score(gt_instances, pred_instances):
    gt_bins = instances_to_text_bins(gt_instances)
    pred_bins = instances_to_text_bins(pred_instances)
    all_bin_keys = set(gt_bins.keys()) | set(pred_bins.keys())
    if not all_bin_keys:
        return {k: 1.0 for k in [
            "exist_all","exist_active","count_all","count_active","count_both",
            "text_all","text_active","text_both"]}
    max_bin = max(all_bin_keys)
    exist_all, count_all, text_all = [], [], []
    exist_active, count_active, text_active = [], [], []
    count_both, text_both = [], []
    for b in range(max_bin + 1):
        gt_t = gt_bins.get(b, [])
        pred_t = pred_bins.get(b, [])
        e, c, t, kind = bin_scores(gt_t, pred_t)
        exist_all.append(e); count_all.append(c); text_all.append(t)
        if kind != "silent":
            exist_active.append(e); count_active.append(c); text_active.append(t)
            if kind == "both":
                count_both.append(c); text_both.append(t)
    def m(arr):
        return float(np.mean(arr)) if arr else 1.0
    return {
        "exist_all": m(exist_all), "exist_active": m(exist_active),
        "count_all": m(count_all), "count_active": m(count_active), "count_both": m(count_both),
        "text_all": m(text_all), "text_active": m(text_active), "text_both": m(text_both),
    }

def compute_one_pair(args):
    channel, video, n_gt, gt_instances, orig_instances, swap_instances = args
    mo = compute_video_score(gt_instances, orig_instances)
    ms = compute_video_score(gt_instances, swap_instances)
    row = {"channel": channel, "video": video, "n_gt": n_gt}
    for key in mo:
        row[f"orig_{key}"] = mo[key]
        row[f"swap_{key}"] = ms[key]
    return row


# ═══════════════════════════════════════
# 1) Pred 수집 (orig + swap)
# ═══════════════════════════════════════
orig_map, swap_map = {}, {}
for path in tqdm(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True), desc="Pred 수집"):
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except:
        continue
    key = (rec["orig_channel"], rec["video_name"])
    instances = rec.get("instances", [])
    if rec.get("orig_channel") == rec.get("cond_channel"):
        orig_map[key] = instances
    else:
        swap_map[key] = instances

# ═══════════════════════════════════════
# 2) GT 수집 + task 구성 (JSONL 줄 단위 순회)
# ═══════════════════════════════════════
tasks = []
with open(EVAL_JSONL, "r") as f:
    for line in f:
        ex = json.loads(line)
        msgs = ex["messages"]
        user_msg = next(m for m in msgs if m["role"] == "user")
        asst_msg = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst_msg is None:
            continue
        user_text = user_msg["content"] if isinstance(user_msg["content"], str) else ""
        m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
        m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
        if not m_ch or not m_vn:
            continue
        key = (m_ch.group(1).strip(), m_vn.group(1).strip())
        if key not in orig_map or key not in swap_map:
            continue
        try:
            gt_instances = json.loads(asst_msg["content"]).get("instances", [])
        except:
            gt_instances = []
        tasks.append((key[0], key[1], len(gt_instances),
                       gt_instances, orig_map[key], swap_map[key]))

print(f"🎯 {len(tasks):,}개 영상 평가")

# ═══════════════════════════════════════
# 3) 병렬 실행
# ═══════════════════════════════════════
results = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(compute_one_pair, t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures), desc="메트릭 계산"):
        results.append(f.result())

print(f"\n✅ 완료: {len(results):,}개")

# ═══════════════════════════════════════
# Δ 계산 + Wilcoxon
# ═══════════════════════════════════════
metric_keys = [
    ("존재 전체", "exist_all"), ("존재 활성", "exist_active"),
    ("개수 전체", "count_all"), ("개수 활성", "count_active"), ("개수 양쪽", "count_both"),
    ("텍스트 전체", "text_all"), ("텍스트 활성", "text_active"), ("텍스트 양쪽", "text_both"),
]

deltas = {}
for label, key in metric_keys:
    deltas[label] = np.array([r[f"orig_{key}"] - r[f"swap_{key}"] for r in results])

print(f"\n📊 상대적 차이 (Δ = orig - swap, 양수면 orig가 나음)")
print(f"  {'메트릭':<15} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8} {'Wilcoxon p':>12}")
print(f"  {'─'*15} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")

for label, delta in deltas.items():
    pos_pct = (delta > 0).sum() / len(delta) * 100
    nonzero = delta[delta != 0]
    if len(nonzero) > 10:
        _, p_val = wilcoxon(nonzero)
        p_str = f"{p_val:.2e}"
    else:
        p_str = "N/A"
    print(f"  {label:<15} {delta.mean():>+10.4f} {np.median(delta):>+10.4f} {pos_pct:>7.1f}% {p_str:>12}")

# ── orig vs swap 상세 ──
print(f"\n📊 orig vs swap 상세 (mean)")
print(f"  {'메트릭':<15} {'orig':>10} {'swap':>10}")
print(f"  {'─'*15} {'─'*10} {'─'*10}")
for label, key in metric_keys:
    o = np.array([r[f"orig_{key}"] for r in results])
    s = np.array([r[f"swap_{key}"] for r in results])
    print(f"  {label:<15} {o.mean():>10.3f} {s.mean():>10.3f}")

# ── GT 인스턴스 구간별 Δ (주요 지표) ──
gt_ns = np.array([r["n_gt"] for r in results])

for label in ["존재 활성", "개수 활성", "텍스트 활성"]:
    delta = deltas[label]
    print(f"\n📊 GT 인스턴스 구간별 Δ {label} (orig - swap)")
    print(f"  {'GT 구간':<15} {'n':>6} {'Δ mean':>10} {'Δ med':>10} {'Δ>0 %':>8}")
    print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")
    for lo, hi in [(0,1),(1,10),(10,100),(100,1000),(1000,10000)]:
        mask = (gt_ns >= lo) & (gt_ns < hi)
        if mask.sum() == 0:
            continue
        d = delta[mask]
        pos = (d > 0).sum() / len(d) * 100
        print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {pos:>7.1f}%")

Pred 수집: 100%|██████████| 6630/6630 [00:01<00:00, 6294.55it/s]


🎯 3,320개 영상 평가


메트릭 계산: 100%|██████████| 3320/3320 [00:06<00:00, 488.36it/s]


✅ 완료: 3,320개

📊 상대적 차이 (Δ = orig - swap, 양수면 orig가 나음)
  메트릭                 Δ mean   Δ median    Δ>0 %   Wilcoxon p
  ─────────────── ────────── ────────── ──────── ────────────
  존재 전체              +0.1274    +0.0000    40.8%     3.47e-83
  존재 활성              +0.0691    +0.0000    31.5%     1.08e-22
  개수 전체              +0.1976    +0.1094    67.7%    8.72e-175
  개수 활성              +0.1389    +0.0280    58.7%    2.48e-121
  개수 양쪽              +0.1265    +0.0373    57.0%    1.43e-129
  텍스트 전체             +0.2323    +0.1349    73.8%    1.17e-255
  텍스트 활성             +0.1731    +0.0579    63.0%    1.72e-220
  텍스트 양쪽             +0.1951    +0.0880    64.1%    4.64e-210

📊 orig vs swap 상세 (mean)
  메트릭                   orig       swap
  ─────────────── ────────── ──────────
  존재 전체                0.829      0.702
  존재 활성                0.712      0.643
  개수 전체                0.631      0.433
  개수 활성                0.511      0.373
  개수 양쪽                0.754      0.627
  텍스트 전체          